## 08. infected cell annotations

In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

# Global plotting configurations for publication-grade vector graphics
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.sans-serif'] = 'Arial'

data_path = "/mnt/d/WY_single_cell_analysis/data/proceed/scVelo/adata_HNRNP_velocity_final.h5ad"
base_dir = "/mnt/d/WY_single_cell_analysis/results/final_annotation/annotation plt"
os.makedirs(base_dir, exist_ok=True)

# Step 1.1: Purge corrupted dendrogram keys from metadata structure
try:
    with h5py.File(data_path, 'r+') as f:
        if 'uns/dendrogram_leiden' in f:
            del f['uns/dendrogram_leiden']
            print("Successfully removed corrupted dendrogram_leiden cache.")
except Exception as e:
    print(f"Cleanup check completed or not required: {e}")

# Step 1.2: Load H5AD dataset
try:
    adata = sc.read_h5ad(data_path)
    print("Data loaded successfully.")
    print(f"Current initial clusters: {list(adata.obs['cell_type'].unique())}")
except Exception as e:
    print(f"Failed to load dataset: {e}")

# Step 1.3: Generate cross-tabulation and composition statistics
group_key = 'Group' 
cluster_key = 'cell_type'

count_df = pd.crosstab(adata.obs[cluster_key], adata.obs[group_key])
total_counts = count_df.sum(axis=0)
ratio_df = count_df.div(total_counts, axis=1) * 100

print("\n--- Absolute Cell Counts per Group ---")
print(count_df)
print("\n--- Cell Proportions per Group (%) ---")
print(ratio_df.round(2))

# Step 1.4: Define precise lineage mapping dictionary
final_refine_map = {
    'Forebrain RG/NPCs': 'Forebrain RG',
    'Homeostatic NPCs': 'Homeostatic NPCs',        
    'Newborn Neurons': 'Newborn Neurons',
    'Intermediate Progenitors': 'IPCs',
    'Cycling NPCs': 'Cycling NPCs',
    'Quiescent RG': 'Quiescent RG',
    'Brain Fibroblasts': 'Brain Fibroblasts',
    'Head neural crest cells': 'Neural Crest',
    'ZIKV-Infected (Type 1)': 'Infected RG',        # GPC3+/ISG15+
    'ZIKV-Infected (Type 2)': 'Infected IPCs',       # EOMES+/ISG15+
    'Infected Neurons (ISG-high)': 'Infected Neurons', # DCX+/ISG15+
    'ER-Stress Cells (CHOP+)': 'Stress Progenitors', # GPC3+/DDIT3+
    'ER-Stress Cells (ATF3+)': 'Apoptotic Cells',    # ATF3+/CASP3+ terminal stress group
    'Ribosomal-active Cells': 'Active-state RG'      # GPC3+/High Translation state
}

# Apply re-annotation mapping and format categories
adata.obs['cell_type_final'] = adata.obs['cell_type'].map(final_refine_map).astype(str).astype('category')

# Step 1.5: Nature-style discrete palette implementation
final_colors = {
    'Homeostatic NPCs': '#749B86', 
    'Forebrain RG': '#99B8C0',     
    'Quiescent RG': '#6DAABD',     
    'Active-state RG': '#D6E9C6',  
    'Infected RG': '#7D4E91',      
    'Infected IPCs': '#4c347c',    
    'Infected Neurons': '#20458C', 
    'Newborn Neurons': '#5A93C1',  
    'IPCs': '#F39C12',             
    'Cycling NPCs': '#E74C3C',     
    'Apoptotic Cells': '#27404E',  
    'Stress Progenitors': '#96646C',
    'Neural Crest': '#F1C40F',     
    'Brain Fibroblasts': '#2C3E50' 
}

# Match color ordering exactly to current categories
adata.uns['cell_type_final_colors'] = [final_colors.get(cat, '#D3D3D3') for cat in adata.obs['cell_type_final'].cat.categories]

# Step 1.6: Plot and export global UMAP with customized minimal L-axis
fig, ax = plt.subplots(figsize=(7, 6))
sc.pl.umap(
    adata, color='cell_type_final', palette=final_colors, size=10, 
    legend_loc='on data', legend_fontsize=10, legend_fontoutline=2,
    frameon=True, ax=ax, show=False
)

# Format axes to an L-shaped structure
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('UMAP1', loc='left', fontsize=12, labelpad=5)
ax.set_ylabel('UMAP2', loc='bottom', fontsize=12, labelpad=5)

save_path_umap = os.path.join(base_dir, "ZIKV_UMAP_L_axis.pdf")
plt.savefig(save_path_umap, format='pdf', bbox_inches='tight', transparent=True)
plt.close()
print(f"Global UMAP saved: {save_path_umap}")

# Step 1.7: Subset metadata columns and preserve file footprint
desired_columns = ['cell_type_final', 'n_genes', 'n_counts', 'n_genes_by_counts', 'total_counts', 'pct_counts_mt']
existing_columns = [col for col in desired_columns if col in adata.obs.columns]
adata_save = adata.obs[existing_columns].copy()

save_obj_dir = "/mnt/d/WY_single_cell_analysis/results/final_annotation/adata_objects/"
os.makedirs(save_obj_dir, exist_ok=True)
save_obj_path = os.path.join(save_obj_dir, "ZIKV_Organoid_Final_Locked.h5ad")

try:
    adata.write_h5ad(save_obj_path)
    print(f"AnnData object successfully synchronized and locked: {save_obj_path}")
except Exception as e:
    print(f"Object serialization failed: {e}")